# 面试问题：GRPO 与 PPO 有什么区别？Group-relative advantage、KL 和 clip 怎样实现？

**一句话回答**：GRPO 对同一 prompt 采样一组 responses，用组内 reward 的相对位置构造 advantage，从而省去单独 value model；随后仍用 old policy ratio、clip 和 reference KL 更新 policy。它减少 critic 内存，却依赖组内可比较奖励、足够多样性和严格的 prompt-group 数据合同。

本 Notebook 手写 group normalization、零方差处理、masked ratio loss、reference KL、staleness 门禁、一次 PyTorch 更新及 group-level 评测，并说明不同论文/实现的 KL 放置可能不同。


In [ ]:
from dataclasses import dataclass
import math
import torch

SEED134=13401; torch.manual_seed(SEED134)
assert SEED134==13401
assert torch.isfinite(torch.randn(4)).all()
assert math.isclose((1+2+3)/3,2)


## 1. 一个训练单元是同 prompt 的完整 response group

所有候选必须由同一 old policy/checkpoint、采样配置和 reward 版本产生；不能在 minibatch shuffle 后把不同 prompt 的 reward 混合标准化。每组还要保存 response mask 与终止原因，超时/格式错误是否给零分必须预先定义。


In [ ]:
@dataclass(frozen=True)
class Sample134:
    prompt_id:str; response_id:str; reward:float; policy_rev:str
group134=[Sample134("p1",f"r{i}",r,"old-7") for i,r in enumerate([0.,1.,1.,3.])]
assert len({x.prompt_id for x in group134})==1
assert len({x.policy_rev for x in group134})==1
assert len({x.response_id for x in group134})==4


## 2. Group-relative advantage 对 reward 平移不敏感

常见形式 `A_i=(r_i-mean(r))/(std(r)+ε)`，同一个 response 的 advantage 会随组内对手改变。它消除了绝对 reward baseline，却不消除 reward 模型偏差。标准差采用 population 还是 sample 口径必须固定，否则小组训练会出现版本差异。


In [ ]:
def group_adv134(rewards,eps=1e-6):
    r=torch.as_tensor(rewards,dtype=torch.float32); return (r-r.mean())/(r.std(unbiased=False)+eps)
a134=group_adv134([0.,1.,1.,3.]); shifted134=group_adv134([10.,11.,11.,13.])
assert torch.allclose(a134,shifted134)
assert abs(float(a134.mean()))<1e-6
assert a134[-1]>0>a134[0]


## 3. 零方差组没有相对学习信号

如果一组全部通过或全部失败，标准化后 advantage 应为零并跳过 policy 更新，而不是除零产生 NaN。治理方法包括提高采样温度、增加 group size、设计更细粒度 verifiable reward，或把难度采样集中到当前策略成功率中间区域。


In [ ]:
flat134=group_adv134([1.,1.,1.,1.])
def informative134(a,tol=1e-5): return float(a.abs().max())>tol
assert torch.allclose(flat134,torch.zeros(4))
assert not informative134(flat134)
assert informative134(a134)


## 4. PPO-style ratio 仍以 rollout old policy 为分母

对 response token 计算 `exp(logπθ-logπold)`，把该 response 的组内 advantage 广播到有效 token；逐 token clip 后按每条 response 或全体 token 聚合。长度归一方式会影响长答案权重，必须明确并做 ablation。


In [ ]:
def grpo_loss134(new_logp,old_logp,adv,mask,eps=.2):
    ratio=torch.exp(new_logp-old_logp); per=torch.minimum(ratio*adv[:,None],ratio.clamp(1-eps,1+eps)*adv[:,None])*mask
    per_response=per.sum(1)/mask.sum(1).clamp_min(1); return -per_response.mean(),ratio
old134=torch.tensor([[-.5,-.4,0.],[-.6,-.2,0.],[-.7,-.3,0.],[-.4,-.1,0.]])
mask134=torch.tensor([[1.,1.,0.]]*4); loss134,ratio134=grpo_loss134(old134+.1,old134,a134,mask134)
assert torch.isfinite(loss134)
assert ratio134.shape==(4,3)
assert torch.allclose(ratio134,torch.full_like(ratio134,math.exp(.1)))


## 5. KL 约束保护 reference 能力，但估计方式要说清楚

可把 reference KL 作为 reward shaping，也可直接加到 loss；不同 GRPO 变体公式不同。完整 KL 需要全词表分布，采样 token 的 log-ratio 是近似监控量。下面实现离散分布精确 `KL(π||π_ref)`，验证非负和同分布为零。


In [ ]:
def exact_kl134(logits,ref_logits):
    lp=logits-torch.logsumexp(logits,-1,keepdim=True); lr=ref_logits-torch.logsumexp(ref_logits,-1,keepdim=True); p=lp.exp(); return (p*(lp-lr)).sum(-1)
z134=torch.tensor([[1.,2.,0.],[0.,0.,0.]])
kl_same134=exact_kl134(z134,z134); kl_move134=exact_kl134(z134,z134+torch.tensor([[1.,-1.,0.],[0.,2.,0.]]))
assert torch.allclose(kl_same134,torch.zeros(2),atol=1e-7)
assert torch.all(kl_move134>=-1e-7)
assert kl_move134.mean()>0


## 6. 手写一步更新验证梯度方向和 mask

Policy 参数更新时 old/ref 都冻结；同一 rollout 复用有限 epoch。这里直接让可学习 log-prob 偏移参与 surrogate，并手工 SGD。真实模型必须从 logits 归一化后 gather，不能把独立 log-prob 参数当生产 policy。


In [ ]:
delta134=torch.nn.Parameter(torch.zeros_like(old134)); before134=delta134.detach().clone()
train_loss134,_=grpo_loss134(old134+delta134,old134,a134.detach(),mask134); train_loss134.backward()
with torch.no_grad(): delta134-=.05*delta134.grad
assert not torch.allclose(delta134,before134)
assert torch.isfinite(delta134).all()
assert torch.all(delta134[:,2]==0)


## 7. Off-policy staleness 与重复样本会破坏组比较

若 rollout policy 落后当前模型太多，ratio 大量触发 clip，样本几乎不再提供有效梯度。用 policy revision、生成时间和近似 KL 做准入；同组响应若大量完全重复，有效 group size 远小于表面数量，也应降权或重采样。


In [ ]:
def usable_group134(policy_revs,responses,expected_rev,min_unique=2):
    return all(x==expected_rev for x in policy_revs) and len(set(responses))>=min_unique
assert usable_group134(["v3"]*4,["a","b","b","c"],"v3")
assert not usable_group134(["v2","v3"],["a","b"],"v3")
assert not usable_group134(["v3"]*3,["a"]*3,"v3")


## 8. 评测单位应是 prompt group，而非只看平均 reward

监控 pass@1/pass@k、组内 reward std、唯一响应率、clip fraction、KL、长度、安全和外部 verifier 准确率；按难度 slice 看是否只优化容易题。若训练 reward 上升而隐藏测试或人工正确率下降，应优先怀疑 reward hacking 和数据污染。


In [ ]:
groups134=[[0,0,1,0],[1,1,1,1],[0,0,0,0]]
pass1_134=sum(g[0]>0 for g in groups134)/len(groups134); passk134=sum(max(g)>0 for g in groups134)/len(groups134)
stds134=[float(torch.tensor(g,dtype=torch.float32).std(unbiased=False)) for g in groups134]
assert passk134>=pass1_134
assert stds134[0]>0 and stds134[1]==0
assert math.isclose(passk134,2/3)


## 面试总结

完整回答是：**同 prompt 同 old-policy 采样 group → verifiable/reward-model 打分 → 组内标准化 advantage → 零方差组跳过 → advantage 广播到 response mask → old-policy ratio + clip → 明确 reference KL 放置 → 有限 epoch 和 staleness 门禁 → unique response/pass@k/clip/KL/隐藏集联合评测**。GRPO 省掉 value model，但把稳定性压力转移到 group 和 reward 设计。

延伸阅读：[DeepSeekMath / GRPO](https://arxiv.org/abs/2402.03300)、[DeepSeek-R1](https://arxiv.org/abs/2501.12948)、[PPO](https://arxiv.org/abs/1707.06347)。
